# Plot a WritingRing recording over a Board timestamp interval

This notebook selects one recording by `user/action/dataset_id`, loads only `ring_0`, retains Board chunks in numeric chunk-index order, and plots Board contacts over an explicit inclusive Board timestamp interval. The separate Ring acceleration plot uses the raw Ring timestamp as its x-axis and annotates events from the matching `timestamp.txt`. Gravity removal is optional and remains assumption-labelled and provisional when automatic stationary calibration does not pass.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.figure import Figure
import numpy as np
import pandas as pd
from IPython.display import display

from writingring import (
    BoardData,
    GravityRemovalConfig,
    GravityRemovalResult,
    RingData,
    discover_recordings,
    load_board,
    load_ring,
    process_ring_gravity,
    select_recording,
)

## Configuration

`BOARD_START_TIMESTAMP` and `BOARD_END_TIMESTAMP` are mandatory absolute Board timestamp values. They are compared only with each Board frame's stored `frame_timestamp_raw`; marker timestamps do not choose the Board interval. The defaults cover the valid Board sequence in `user_0/action 0/dataset 0`.

In [ ]:
USER = "user_0"
ACTION = "0"
DATASET_ID = 0

BOARD_START_TIMESTAMP = 1720476262355268
BOARD_END_TIMESTAMP = 1720476305908521

REMOVE_GRAVITY = False
GRAVITY_CONFIG = GravityRemovalConfig(
    sampling_rate_hz=200.0,
    acceleration_scale_to_working_units=1.0,
    acceleration_unit_label="m/s^2",
    gyro_scale_to_rad_s=1.0,
    strict_calibration=False,
)

## Resolve paths and select one recording

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root containing pyproject.toml and data/")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "recording_interval"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

recordings = discover_recordings(DATA_ROOT)
recording = select_recording(
    recordings,
    user=USER,
    action=ACTION,
    dataset_id=DATASET_ID,
)

if recording.timestamp_path is None:
    raise FileNotFoundError(f"Selected recording has no timestamp.txt: {recording}")
if not isinstance(BOARD_START_TIMESTAMP, int) or isinstance(BOARD_START_TIMESTAMP, bool):
    raise TypeError("BOARD_START_TIMESTAMP must be an integer Board timestamp")
if not isinstance(BOARD_END_TIMESTAMP, int) or isinstance(BOARD_END_TIMESTAMP, bool):
    raise TypeError("BOARD_END_TIMESTAMP must be an integer Board timestamp")
if BOARD_START_TIMESTAMP > BOARD_END_TIMESTAMP:
    raise ValueError("BOARD_START_TIMESTAMP must be <= BOARD_END_TIMESTAMP")

chunk_indices = recording.board_chunk_indices
if chunk_indices != tuple(sorted(chunk_indices)):
    raise AssertionError(f"Board chunks are not in numeric order: {chunk_indices}")

print(f"Data root: {DATA_ROOT}")
print(f"Selected: {recording.user} / action {recording.action} / dataset {recording.dataset_id}")
print(f"Ring payload that will be loaded: {recording.ring_0_path.name}")
print(f"Ring 1 metadata only (never loaded): {recording.ring_1_path.name if recording.ring_1_path else 'absent'}")
print(f"Timestamp markers: {recording.timestamp_path.name}")
print(f"Numeric Board chunk indices: {chunk_indices}")
print(f"Board interval (inclusive): [{BOARD_START_TIMESTAMP}, {BOARD_END_TIMESTAMP}]")

## Load Ring, Board, and marker data

The reusable loaders preserve raw values and report anomalies. `load_ring(recording)` always resolves to `ring_0`; `ring_1` remains metadata only.

In [ ]:
def load_markers(path: Path) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for line_number, raw_line in enumerate(path.read_text(encoding="utf-8").splitlines(), start=1):
        line = raw_line.strip()
        if not line:
            continue
        parts = line.split(maxsplit=1)
        if len(parts) != 2 or not parts[1].strip():
            raise ValueError(f"Malformed marker line {line_number} in {path}: {raw_line!r}")
        try:
            timestamp = int(parts[0])
        except ValueError as error:
            raise ValueError(
                f"Marker timestamp on line {line_number} is not an integer: {parts[0]!r}"
            ) from error
        rows.append({"marker_index": len(rows), "timestamp": timestamp, "label": parts[1].strip()})

    markers = pd.DataFrame(rows, columns=("marker_index", "timestamp", "label"))
    if markers.empty:
        raise ValueError(f"Marker file is empty: {path}")
    if not markers["timestamp"].is_monotonic_increasing or markers["timestamp"].duplicated().any():
        raise ValueError(f"Marker timestamps must be strictly increasing: {path}")
    return markers


ring_data = load_ring(recording)
board_data = load_board(recording)
markers = load_markers(recording.timestamp_path)

assert ring_data.source_path == recording.ring_0_path
assert ring_data.source_path.name == f"{DATASET_ID}_ring_0.bin"
assert board_data.chunk_paths == recording.board_chunk_paths
assert tuple(report.chunk_index for report in board_data.chunk_reports) == chunk_indices

chunk_table = pd.DataFrame(
    {
        "chunk_index": [report.chunk_index for report in board_data.chunk_reports],
        "frame_count": [report.frame_count for report in board_data.chunk_reports],
        "first_frame_timestamp": [report.first_frame_timestamp for report in board_data.chunk_reports],
        "last_frame_timestamp": [report.last_frame_timestamp for report in board_data.chunk_reports],
        "empty": [report.empty for report in board_data.chunk_reports],
        "warnings": ["; ".join(report.warnings) for report in board_data.chunk_reports],
    }
)
display(chunk_table)
display(markers)
if board_data.warnings:
    print("Board warnings:")
    for warning in board_data.warnings:
        print(f"- {warning}")

## Board timestamp interval

Frames and contacts are filtered independently with the same inclusive raw Board timestamp bounds. The source `BoardData` tables remain unchanged.

In [ ]:
board_frames_interval = board_data.frames.loc[
    board_data.frames["frame_timestamp_raw"].between(
        BOARD_START_TIMESTAMP, BOARD_END_TIMESTAMP, inclusive="both"
    )
].copy()
board_contacts_interval = board_data.contacts.loc[
    board_data.contacts["frame_timestamp_raw"].between(
        BOARD_START_TIMESTAMP, BOARD_END_TIMESTAMP, inclusive="both"
    )
].copy()

if board_frames_interval.empty:
    raise ValueError(
        "No Board frames fall inside the configured timestamp interval "
        f"[{BOARD_START_TIMESTAMP}, {BOARD_END_TIMESTAMP}]"
    )

print(f"Selected Board frames: {len(board_frames_interval):,}")
print(f"Selected Board contacts: {len(board_contacts_interval):,}")
print(
    "Observed selected frame timestamp range: "
    f"{int(board_frames_interval['frame_timestamp_raw'].min())} .. "
    f"{int(board_frames_interval['frame_timestamp_raw'].max())}"
)
display(board_frames_interval.head())
display(board_contacts_interval.head())

## Board trajectory and contact-force plots

The pressure plot uses the verified per-contact `force` value. Its physical unit is not documented, so the axis is labelled as a raw value.

In [ ]:
def plot_board_trajectory_interval(
    contacts: pd.DataFrame,
    *,
    start_timestamp: int,
    end_timestamp: int,
    output_path: Path,
) -> Figure:
    figure, axis = plt.subplots(figsize=(8, 7), layout="constrained")
    if contacts.empty:
        axis.text(0.5, 0.5, "No contacts in the selected Board interval", ha="center", va="center", transform=axis.transAxes)
    else:
        points = axis.scatter(
            contacts["x"].to_numpy(copy=True),
            contacts["y_display"].to_numpy(copy=True),
            c=contacts["force"].to_numpy(copy=True),
            s=14,
            alpha=0.8,
            cmap="viridis",
        )
        figure.colorbar(points, ax=axis, label="Contact force (raw value; units undocumented)")
    axis.set_xlabel("Stored normalized x")
    axis.set_ylabel("Display y = 1 - y_raw")
    axis.set_aspect("equal", adjustable="box")
    axis.grid(True, alpha=0.3)
    axis.set_title(
        f"Board contact trajectory — {USER}/action {ACTION}/dataset {DATASET_ID}\n"
        f"raw Board timestamp [{start_timestamp}, {end_timestamp}]"
    )
    figure.savefig(output_path, dpi=150)
    return figure


def plot_board_force_interval(
    contacts: pd.DataFrame,
    *,
    start_timestamp: int,
    end_timestamp: int,
    output_path: Path,
) -> Figure:
    figure, axis = plt.subplots(figsize=(13, 5), layout="constrained")
    if contacts.empty:
        axis.text(0.5, 0.5, "No contact force values in the selected Board interval", ha="center", va="center", transform=axis.transAxes)
    else:
        axis.scatter(
            contacts["frame_timestamp_raw"].to_numpy(copy=True),
            contacts["force"].to_numpy(copy=True),
            s=10,
            alpha=0.7,
        )
    axis.set_xlim(start_timestamp, end_timestamp)
    axis.set_xlabel("Raw Board frame timestamp (stored value)")
    axis.set_ylabel("Contact force (raw value; units undocumented)")
    axis.grid(True, alpha=0.3)
    axis.set_title(
        f"Board contact force — {USER}/action {ACTION}/dataset {DATASET_ID}\n"
        f"raw Board timestamp [{start_timestamp}, {end_timestamp}]"
    )
    figure.savefig(output_path, dpi=150)
    return figure


trajectory_path = OUTPUT_DIR / f"{USER}_action-{ACTION}_dataset-{DATASET_ID}_board_trajectory.png"
force_path = OUTPUT_DIR / f"{USER}_action-{ACTION}_dataset-{DATASET_ID}_board_force.png"

trajectory_figure = plot_board_trajectory_interval(
    board_contacts_interval,
    start_timestamp=BOARD_START_TIMESTAMP,
    end_timestamp=BOARD_END_TIMESTAMP,
    output_path=trajectory_path,
)
display(trajectory_figure)
plt.close(trajectory_figure)

force_figure = plot_board_force_interval(
    board_contacts_interval,
    start_timestamp=BOARD_START_TIMESTAMP,
    end_timestamp=BOARD_END_TIMESTAMP,
    output_path=force_path,
)
display(force_figure)
plt.close(force_figure)

## Ring acceleration with timestamp markers

The x-axis is the unchanged raw Ring timestamp. Marker lines come only from the matching `timestamp.txt`. When gravity removal is enabled, processing uses a fixed nominal `dt = 1 / sampling_rate_hz`, not differences between the duplicate-containing raw Ring timestamps. Automatic calibration failures continue only because `strict_calibration=False`; such output is explicitly marked provisional.

In [ ]:
gravity_result: GravityRemovalResult | None = None
if REMOVE_GRAVITY:
    gravity_result = process_ring_gravity(ring_data, config=GRAVITY_CONFIG)
    calibration = gravity_result.calibration
    search = calibration.stationary_search
    gravity_summary = pd.Series(
        {
            "provisional": gravity_result.diagnostics.provisional,
            "calibration_passed": calibration.passed,
            "calibration_start_sample": calibration.start_sample,
            "calibration_stop_sample": calibration.stop_sample,
            "gravity_magnitude": calibration.gravity_magnitude,
            "stationary_search_passed": None if search is None else search.passed,
            "stationary_search_score": None if search is None else search.score,
            "failed_checks": None if search is None else search.failed_checks,
            "nominal_sampling_rate_hz": gravity_result.diagnostics.nominal_sampling_rate_hz,
            "warnings": gravity_result.diagnostics.warnings,
        },
        name="Gravity-removal diagnostics",
    )
    display(gravity_summary)
else:
    print("Gravity removal disabled: plotting unchanged raw acc_x/acc_y/acc_z.")

In [ ]:
def plot_ring_acceleration_with_markers(
    ring: RingData,
    marker_table: pd.DataFrame,
    *,
    gravity: GravityRemovalResult | None,
    output_path: Path,
) -> Figure:
    timestamps = ring.dataframe["timestamp"].to_numpy(copy=True)
    if gravity is None:
        acceleration = ring.dataframe.loc[:, ["acc_x", "acc_y", "acc_z"]].to_numpy(copy=True)
        legend_labels = ("acc_x", "acc_y", "acc_z")
        y_label = "Raw acceleration value (units undocumented)"
        mode_title = "raw acceleration"
        warning_line = "Ring timestamp unit/origin is inferred, not independently documented"
    else:
        acceleration = gravity.linear_acceleration_body
        legend_labels = ("linear_acc_body_x", "linear_acc_body_y", "linear_acc_body_z")
        y_label = f"Estimated linear acceleration ({gravity.config.acceleration_unit_label})"
        mode_title = "estimated gravity-removed acceleration"
        status = "PROVISIONAL" if gravity.diagnostics.provisional else "calibration passed"
        warning_line = (
            f"{status}; fixed nominal rate={gravity.config.sampling_rate_hz:g} Hz; "
            f"profile={gravity.config.profile_name}; noncausal offline estimate"
        )

    figure, axis = plt.subplots(figsize=(15, 6), layout="constrained")
    for axis_index, label in enumerate(legend_labels):
        axis.plot(timestamps, acceleration[:, axis_index], label=label, linewidth=0.9)

    visible_markers = marker_table.loc[
        marker_table["timestamp"].between(float(timestamps[0]), float(timestamps[-1]), inclusive="both")
    ]
    for row in visible_markers.itertuples(index=False):
        axis.axvline(row.timestamp, color="tab:red", alpha=0.35, linewidth=0.8)
        axis.text(
            row.timestamp,
            0.98,
            str(row.label),
            transform=axis.get_xaxis_transform(),
            rotation=90,
            ha="right",
            va="top",
            fontsize=8,
            color="tab:red",
        )

    axis.set_xlabel("Raw Ring IMU timestamp (stored value)")
    axis.set_ylabel(y_label)
    axis.grid(True, alpha=0.3)
    axis.legend(loc="lower right")
    axis.set_title(
        f"Ring 0 {mode_title} — {USER}/action {ACTION}/dataset {DATASET_ID}\n"
        f"{warning_line}"
    )
    figure.savefig(output_path, dpi=150)
    return figure


if gravity_result is None:
    ring_mode = "raw_accel"
elif gravity_result.diagnostics.provisional:
    ring_mode = "linear_accel_provisional"
else:
    ring_mode = "linear_accel"
ring_accel_path = OUTPUT_DIR / f"{USER}_action-{ACTION}_dataset-{DATASET_ID}_ring_{ring_mode}_markers.png"
ring_figure = plot_ring_acceleration_with_markers(
    ring_data,
    markers,
    gravity=gravity_result,
    output_path=ring_accel_path,
)
display(ring_figure)
plt.close(ring_figure)

## Execution checks and limitations

In [ ]:
assert ring_data.source_path.name.endswith("_ring_0.bin")
assert chunk_indices == tuple(sorted(chunk_indices))
assert board_frames_interval["frame_timestamp_raw"].between(
    BOARD_START_TIMESTAMP, BOARD_END_TIMESTAMP, inclusive="both"
).all()
assert board_contacts_interval["frame_timestamp_raw"].between(
    BOARD_START_TIMESTAMP, BOARD_END_TIMESTAMP, inclusive="both"
).all()
assert list(markers.columns) == ["marker_index", "timestamp", "label"]
assert all(path.is_file() and path.stat().st_size > 0 for path in (trajectory_path, force_path, ring_accel_path))
if gravity_result is not None:
    assert gravity_result.sample_count == len(ring_data.dataframe)
    assert np.isfinite(gravity_result.linear_acceleration_body).all()

display(
    pd.DataFrame(
        {
            "output_file": [trajectory_path.name, force_path.name, ring_accel_path.name],
            "size_bytes": [trajectory_path.stat().st_size, force_path.stat().st_size, ring_accel_path.stat().st_size],
        }
    )
)
print("Verified: Board filtering used only stored Board frame timestamps.")
print("Verified: timestamp.txt was used only to annotate the Ring acceleration plot.")
print("No Ring–Board synchronization, timestamp repair, or stale-chunk deletion was performed.")

### Interpretation cautions

- Ring timestamp units and clock origin are inferred from the sample, not independently documented.
- Board timestamps are stored per frame and originate from `int(time.time() * 1e6)` in the supplied writer.
- `timestamp.txt` contains sparse character/word event markers, not per-sample timestamps.
- Board contact force units are undocumented.
- Gravity removal assumes acceleration in `m/s^2`, gyroscope in `rad/s`, a 200 Hz nominal rate, and identity sensor axes unless the configuration is deliberately changed.
- Automatic stationary detection proposes a calibration interval but does not prove physical stationarity. A failed search produces a clearly labelled provisional result because this notebook configures `strict_calibration=False`.
- Similar Ring, Board, and marker timestamp magnitudes permit operational alignment but do not establish hardware synchronization.